# Tagalog Fake News Detection with an Affix-Aware Kernel SVM

This notebook trains a support vector machine (SVM) on the Fake News Filipino dataset. It uses a custom kernel that groups affixed forms of the same root, and compares it with standard baselines.

**Steps**

1. Configuration
2. Data loading
3. Preprocessing and train/test split
4. Word counts and TF-IDF
5. Rule-based affix stripper
6. Soft stem map and affix map
7. Gap-weighted subsequence features
8. Document feature map
9. Gram matrices and the weighted-sum kernel
10. Validity (PSD) checks
11. Baselines
12. Cross-validation
13. Final model, ablations and RBF wrapper
14. Results summary

**Method in brief.** The kernel is defined as the inner product of an explicit feature map $\Phi$ with three blocks: stem, affix and character subsequence. Because it is an inner product of feature vectors, it is positive semi-definite (PSD) by construction. The validity checks verify that the implementation matches this.

## Configuration

Imports and hyperparameters.

- `KAGGLE_DIR` is the dataset folder, and `DATA_SOURCE` selects the data source (`"kaggle"`, `"hf"`, `"csv"` or `"toy"`).
- `MIN_DF`, `MIN_ROOT_FREQ`, `MIN_STEM_LEN` and `ALPHA` control the stemmer and the soft stem map.
- `SUBSEQ_P`, `SUBSEQ_LAMBDA` and `MAX_WORD_LEN` control the subsequence features.

In [3]:
import os, re, itertools, time
from collections import Counter, defaultdict

import numpy as np
import scipy.sparse as sp
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
KAGGLE_DIR = os.environ.get("KAGGLE_DIR", "/kaggle/input/datasets/ivancobarrubias/fake-news-filipino")
DATA_SOURCE = os.environ.get("DATA_SOURCE", "kaggle" if os.path.isdir(KAGGLE_DIR) else "hf")
CSV_PATH = None

MIN_DF = 2
MIN_ROOT_FREQ = 3
MIN_STEM_LEN = 3
ALPHA = 1.0
SUBSEQ_P = 3
SUBSEQ_LAMBDA = 0.5
MAX_WORD_LEN = 14

## Data loading

Loads the articles and their labels.

- For `"kaggle"`, `load_from_folder` lists the data files under `KAGGLE_DIR`. It uses a combined file (name starting with `full` or `all`) if one exists, and otherwise merges all files.
- The text column is the one with the longest average length. The label column is `label`, or the best column with few distinct values. Duplicate articles are dropped and labels are encoded as integers.
- The cell prints the files found, the columns chosen and the label mapping. Check these before continuing.
- `"hf"` downloads from the Hugging Face Hub, `"csv"` reads one file from `CSV_PATH`, and `"toy"` generates a small synthetic corpus for testing the pipeline.

In [4]:
def load_from_folder(folder):
    import pandas as pd
    exts = {".csv": pd.read_csv, ".tsv": lambda p: pd.read_csv(p, sep="\t"), ".json": pd.read_json,
            ".jsonl": lambda p: pd.read_json(p, lines=True), ".parquet": pd.read_parquet}
    files = sorted(os.path.join(r, f) for r, _, fs in os.walk(folder) for f in fs if os.path.splitext(f)[1].lower() in exts)
    print("files found under", folder)
    for f in files: print("  ", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")
    if not files:
        raise FileNotFoundError(f"No csv/tsv/json/jsonl/parquet files under {folder}. Check KAGGLE_DIR.")
    full = [f for f in files if os.path.basename(f).lower().startswith(("full", "all"))]
    use = full if full else files
    frames = [exts[os.path.splitext(f)[1].lower()](f) for f in use]
    df = pd.concat(frames, ignore_index=True)
    print("using", [os.path.basename(f) for f in use], "-> columns:", list(df.columns), "| rows:", len(df))
    str_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    text_col = max(str_cols, key=lambda c: df[c].astype(str).str.len().mean())
    cands = [c for c in df.columns if c != text_col and df[c].nunique() <= 10]
    label_col = next((c for c in df.columns if c.lower() in ("label", "labels", "class", "target")), cands[0] if cands else None)
    if label_col is None:
        raise ValueError(f"Could not find a label column among {list(df.columns)}")
    df = df.drop_duplicates(subset=text_col).dropna(subset=[text_col, label_col])
    codes, uniques = pd.factorize(df[label_col], sort=True)
    print(f"text column: '{text_col}' | label column: '{label_col}' | mapping: " + ", ".join(f"{u!r}->{i}" for i, u in enumerate(uniques)))
    return df[text_col].astype(str).tolist(), np.asarray(codes)

def load_texts_labels():
    if DATA_SOURCE == "kaggle":
        return load_from_folder(KAGGLE_DIR)
    if DATA_SOURCE == "hf":
        from datasets import load_dataset
        try:
            ds = load_dataset("jcblaise/fake_news_filipino", trust_remote_code=True)
        except Exception as e:
            raise RuntimeError("Could not load the dataset from the Hub (" + str(e)[:200] + "). Download the raw files from the "
                               "authors' GitHub, build a CSV with columns article,label, set CSV_PATH and DATA_SOURCE='csv'.")
        print(ds)
        texts, labels = [], []
        for split in ds.keys():
            cols = ds[split].column_names
            text_col = "article" if "article" in cols else [c for c in cols if c != "label"][0]
            texts += list(ds[split][text_col])
            labels += list(ds[split]["label"])
        return texts, np.array(labels)
    if DATA_SOURCE == "csv":
        import pandas as pd
        df = pd.read_csv(CSV_PATH)
        return df["article"].astype(str).tolist(), df["label"].to_numpy()
    if DATA_SOURCE == "toy":
        rng = np.random.default_rng(SEED)
        roots_a = ["kain", "sulat", "basa", "alis", "dating", "tulong", "bili", "laro", "aral", "punta"]
        roots_b = ["luto", "inom", "hanap", "bigay", "kuha", "tawag", "ayos", "sabi", "gawa", "dala"]
        def forms(r):
            c = r[0]
            out = [r, c + "um" + r[1:], c + "in" + r[1:], r[:2] + r, r + "an", r + "in", "mag" + r, "pag" + r, "nag" + r[:2] + r]
            return out
        filler = ["ang", "ng", "sa", "ay", "na", "ang", "mga", "para", "ito", "si", "ni", "at"]
        texts, labels = [], []
        for i in range(400):
            y = i % 2
            pool = [f for r in (roots_a if y == 0 else roots_b) for f in forms(r)]
            other = [f for r in (roots_b if y == 0 else roots_a) for f in forms(r)]
            toks = [rng.choice(pool) if rng.random() < 0.35 else (rng.choice(other) if rng.random() < 0.15 else rng.choice(filler))
                    for _ in range(120)]
            texts.append(" ".join(toks) + ".")
            labels.append(y)
        return texts, np.array(labels)
    raise ValueError(DATA_SOURCE)

texts, y_all = load_texts_labels()
print(len(texts), "documents; label counts:", Counter(y_all.tolist()))

files found under /kaggle/input/datasets/ivancobarrubias/fake-news-filipino
   /kaggle/input/datasets/ivancobarrubias/fake-news-filipino/full.csv (3.6 MB)
using ['full.csv'] -> columns: ['label', 'article'] | rows: 3206
text column: 'article' | label column: 'label' | mapping: 0->0, 1->1
3005 documents; label counts: Counter({1: 1509, 0: 1496})


## Preprocessing and train/test split

Tokenizes each article: the text is lowercased, only letters are kept, and hyphens inside words are removed (*mag-text* becomes *magtext*) so that *mag-* is treated as an ordinary prefix. Discarding punctuation, digits and capitalization also removes formatting cues that could reveal the source.

The data are then split into 80% training and 20% test sets, stratified by label, with a fixed random seed.

In [5]:
TOKEN_RE = re.compile(r"[a-zñ]+(?:-[a-zñ]+)*")

def tokenize(text):
    return [t.replace("-", "") for t in TOKEN_RE.findall(text.lower())]

clean = [" ".join(tokenize(t)) for t in texts]
idx_tr, idx_te = train_test_split(np.arange(len(clean)), test_size=0.2, stratify=y_all, random_state=SEED)
docs_tr = [clean[i] for i in idx_tr]; docs_te = [clean[i] for i in idx_te]
y_tr, y_te = y_all[idx_tr], y_all[idx_te]
print("train:", len(docs_tr), " test:", len(docs_te))

train: 2404  test: 601


## Word counts and TF-IDF

- `CountVectorizer` builds the vocabulary (words occurring in at least `MIN_DF` training documents) and the count matrix $n(d)$.
- `TfidfTransformer` (sublinear term frequency, no normalization) produces the weighted vectors $x(d)$. The IDF is estimated on the training set only.
- The corpus frequency of each word is stored in `freq_d`. It is used later to decide which candidate stems are real roots and to weight them.

In [6]:
cv = CountVectorizer(token_pattern=r"\S+", lowercase=False, min_df=MIN_DF)
Xc_tr = cv.fit_transform(docs_tr)
Xc_te = cv.transform(docs_te)
vocab = cv.get_feature_names_out()
V = len(vocab)
w2i = {w: i for i, w in enumerate(vocab)}
freq = np.asarray(Xc_tr.sum(0)).ravel()
freq_d = dict(zip(vocab, freq))

tfidf = TfidfTransformer(norm=None, sublinear_tf=True)
X_tr = tfidf.fit_transform(Xc_tr).tocsr()
X_te = tfidf.transform(Xc_te).tocsr()
print("vocabulary size V =", V)

vocabulary size V = 16401


## Rule-based affix stripper

- `one_step` proposes candidate stems by undoing one affixation process: prefixation, suffixation (-an/-han, -in/-hin), infixation (-um-, -in-), reduplication, and two sound changes (u to o, d to r).
- `derive` applies these rules up to three times (breadth-first). It keeps the word itself and only those candidates that occur at least `MIN_ROOT_FREQ` times in the training text. This filter removes most non-words, but a real word that is a substring of another word can survive it: *pagod*, for example, can be reduced to *god*.
- The final loop prints the raw candidates without the filter for several example words. Non-words such as *gkain* appear there and are removed in the soft stem map step.

In [7]:
PREFIXES = sorted(["mag", "nag", "pag", "ma", "na", "ka", "pa", "i", "ipa", "ipag", "naka", "maka", "maki",
                   "naki", "nagpa", "magpa", "pang", "mang", "nang", "pinag", "pina", "ika", "taga"],
                  key=len, reverse=True)
SUFFIXES = ["han", "hin", "an", "in"]
VOWELS = set("aeiou")

def one_step(x):
    for p in PREFIXES:
        if x.startswith(p) and len(x) - len(p) >= MIN_STEM_LEN:
            yield x[len(p):], "pre:" + p
    for s in SUFFIXES:
        if x.endswith(s) and len(x) - len(s) >= MIN_STEM_LEN:
            r = x[:-len(s)]
            yield r, "suf:" + s
            j = r.rfind("u")
            if j >= max(0, len(r) - 3):
                yield r[:j] + "o" + r[j + 1:], "suf:" + s + "+u>o"
    if len(x) >= MIN_STEM_LEN + 2:
        if x[0] not in VOWELS and x[1:3] in ("um", "in"):
            yield x[0] + x[3:], "inf:" + x[1:3]
        if x[0] in VOWELS and x[:2] in ("um", "in"):
            yield x[2:], "inf:" + x[:2]
    if len(x) >= 4 and x[0] not in VOWELS and x[1] in VOWELS and x[:2] == x[2:4]:
        yield x[2:], "red"
    if len(x) >= 3 and x[0] in VOWELS and x[0] == x[1]:
        yield x[1:], "red"
    if len(x) >= 5 and x[0] == "d" and x[2] == "r" and x[1] == x[3]:
        yield "d" + x[3:], "red:d>r"

def derive(word, is_root, max_depth=3):
    found = {word: ()}
    frontier = [(word, ())]
    for _ in range(max_depth):
        nxt = []
        for x, tags in frontier:
            for c, t in one_step(x):
                if c not in found and len(c) >= MIN_STEM_LEN:
                    found[c] = tags + (t,)
                    nxt.append((c, found[c]))
        frontier = nxt
    return {c: t for c, t in found.items() if c == word or is_root(c)}

for w in ["kumain", "kinakain", "nagkakain", "pagkain", "kainan", "darating", "tulungan", "basahin"]:
    print(f"{w:10s} ->", {c: "+".join(t) or "(self)" for c, t in derive(w, lambda c: True).items()})

kumain     -> {'kumain': '(self)', 'kuma': 'suf:in', 'koma': 'suf:in+u>o', 'kain': 'inf:um'}
kinakain   -> {'kinakain': '(self)', 'kinaka': 'suf:in', 'kakain': 'inf:in', 'kaka': 'suf:in+inf:in', 'kain': 'inf:in+pre:ka'}
nagkakain  -> {'nagkakain': '(self)', 'kakain': 'pre:nag', 'gkakain': 'pre:na', 'nagkaka': 'suf:in', 'kain': 'pre:nag+pre:ka', 'kaka': 'pre:nag+suf:in', 'gkaka': 'pre:na+suf:in'}
pagkain    -> {'pagkain': '(self)', 'kain': 'pre:pag', 'gkain': 'pre:pa', 'pagka': 'suf:in', 'gka': 'pre:pa+suf:in'}
kainan     -> {'kainan': '(self)', 'inan': 'pre:ka', 'kain': 'suf:an', 'nan': 'pre:ka+pre:i'}
darating   -> {'darating': '(self)', 'dating': 'red:d>r'}
tulungan   -> {'tulungan': '(self)', 'tulung': 'suf:an', 'tulong': 'suf:an+u>o'}
basahin    -> {'basahin': '(self)', 'basa': 'suf:hin', 'basah': 'suf:in'}


## Soft stem map and affix map

- `build_maps` assigns each word a distribution over its candidate stems, $P(s \mid w) \propto \mathrm{freq}(s)^{\alpha}$, where $\alpha$ is `ALPHA`. These probabilities form the column-stochastic stem map $M$ with $M_{s,w} = P(s \mid w)$.
- The affix map $A$ records which affix types occur in each derivation, weighted by $P(s \mid w)$.
- Since the word-similarity matrix $S = M^{\top} M$ is PSD, the stem kernel is valid.
- The assertion confirms that each column of $M$ sums to 1. `show_stems` prints the most probable stems for sample words.

In [8]:
def build_maps(vocab, freq_d, alpha=ALPHA, min_root_freq=MIN_ROOT_FREQ):
    is_root = lambda c: freq_d.get(c, 0) >= min_root_freq
    tag_index = {}
    m_r, m_c, m_v = [], [], []
    a_r, a_c, a_v = [], [], []
    deriv_cache = {}
    for j, w in enumerate(vocab):
        cands = derive(w, is_root)
        deriv_cache[w] = cands
        scores = {c: freq_d[c] ** alpha for c in cands}
        Z = sum(scores.values())
        for c, sc in scores.items():
            p = sc / Z
            m_r.append(w2i[c]); m_c.append(j); m_v.append(p)
            for t in cands[c]:
                k = tag_index.setdefault(t, len(tag_index))
                a_r.append(k); a_c.append(j); a_v.append(p)
    M = sp.csr_matrix((m_v, (m_r, m_c)), shape=(len(vocab), len(vocab)))
    A = sp.csr_matrix((a_v, (a_r, a_c)), shape=(max(len(tag_index), 1), len(vocab)))
    return M, A, tag_index, deriv_cache

t0 = time.time()
M, A, tag_index, deriv_cache = build_maps(vocab, freq_d)
print(f"M: {M.shape}, A: {A.shape}, distinct affix tags: {len(tag_index)}, built in {time.time() - t0:.1f}s")
assert np.allclose(np.asarray(M.sum(0)).ravel(), 1.0), "columns of M must sum to 1 (probabilities)"

def show_stems(words, top=3):
    for w in words:
        if w not in w2i:
            print(f"{w:12s} (not in vocabulary)"); continue
        col = M[:, w2i[w]].toarray().ravel()
        best = np.argsort(-col)[:top]
        print(f"{w:12s} ->", ", ".join(f"{vocab[i]} ({col[i]:.2f})" for i in best if col[i] > 0))

show_stems(["kumain", "kinakain", "nagkakain", "pagkain", "kainan", "pagod", "mata"])

M: (16401, 16401), A: (35, 16401), distinct affix tags: 35, built in 0.2s
kumain       -> kumain (0.70), kain (0.30)
kinakain     -> kinakain (0.45), kain (0.27), kakain (0.27)
nagkakain    (not in vocabulary)
pagkain      -> pagkain (0.79), pagka (0.16), kain (0.05)
kainan       -> kain (0.50), kainan (0.50)
pagod        -> god (0.80), pagod (0.20)
mata         -> mata (1.00)


## Gap-weighted subsequence features

- `subseq_features` lists all letter subsequences of length `SUBSEQ_P` in a word (at most `MAX_WORD_LEN` letters), each weighted by $\lambda^{\text{span}}$, where $\lambda$ is `SUBSEQ_LAMBDA` and the span is the distance covered in the word.
- Each word vector is $\ell_2$-normalized, giving the matrix `C_words` (vocabulary by subsequence features).
- The printed check confirms that $\varphi_{\text{kain}}(\text{kumain}) = \lambda^{6}$ and $\varphi_{\text{kain}}(\text{kain}) = \lambda^{4}$ for subsequence length 4: the infixed form *kumain* contains *kain* with a gap, so it keeps a graded similarity to the root.

In [9]:
def subseq_features(word, p=SUBSEQ_P, lam=SUBSEQ_LAMBDA):
    w = word[:MAX_WORD_LEN]
    feats = defaultdict(float)
    for idx in itertools.combinations(range(len(w)), p):
        feats["".join(w[i] for i in idx)] += lam ** (idx[-1] - idx[0] + 1)
    return feats

u_index = {}
c_r, c_c, c_v = [], [], []
for j, w in enumerate(vocab):
    for u, val in subseq_features(w).items():
        k = u_index.setdefault(u, len(u_index))
        c_r.append(j); c_c.append(k); c_v.append(val)
C_words = normalize(sp.csr_matrix((c_v, (c_r, c_c)), shape=(V, len(u_index))))
print("subsequence features U =", len(u_index))

f1, f2 = subseq_features("kumain", p=4), subseq_features("kain", p=4)
print("phi_kain(kumain) =", f1["kain"], "(lambda^6 =", SUBSEQ_LAMBDA ** 6, ")  phi_kain(kain) =", f2["kain"],
      "(lambda^4 =", SUBSEQ_LAMBDA ** 4, ")")

subsequence features U = 12332
phi_kain(kumain) = 0.015625 (lambda^6 = 0.015625 )  phi_kain(kain) = 0.0625 (lambda^4 = 0.0625 )


## Document feature map

`doc_blocks` maps the TF-IDF matrix to the three document-level blocks, where $C$ is the matrix `C_words`:

- stem block: $\Phi_{\text{stem}}(d) = M\,x(d)$
- affix block: $\Phi_{\text{aff}}(d) = A\,x(d)$
- character block: $\Phi_{\text{char}}(d) = C^{\top} x(d)$

Each block is $\ell_2$-normalized per document. The blocks are computed for the training and test sets.

In [10]:
def doc_blocks(X):
    B_stem = normalize(X @ M.T)
    B_aff = normalize(X @ A.T)
    B_char = normalize(X @ C_words)
    return [B_stem.tocsr(), B_aff.tocsr(), B_char.toarray()]

blocks_tr = doc_blocks(X_tr)
blocks_te = doc_blocks(X_te)
for name, b in zip(["stem", "affix", "char"], blocks_tr):
    print(f"{name:6s} block shape {b.shape}")

stem   block shape (2404, 16401)
affix  block shape (2404, 35)
char   block shape (2404, 12332)


## Gram matrices and the weighted-sum kernel

- For each block, the cell computes the train-by-train and test-by-train Gram matrices and the diagonal values.
- `combine` forms $K = a\,K_{\text{stem}} + b\,K_{\text{aff}} + c\,K_{\text{char}}$ with nonnegative weights $a, b, c \ge 0$ (a nonnegative weighted sum of kernels is a kernel) and normalizes it so that $K(d, d) = 1$.
- The default weights are $(a, b, c) = (1, 1, 1)$. Block Gram matrices are computed once, so changing the weights only requires a weighted sum.

In [11]:
def gram(Ba, Bb):
    G = Ba @ Bb.T
    return G.toarray() if sp.issparse(G) else np.asarray(G)

def diag_of(B):
    return np.asarray(B.multiply(B).sum(1)).ravel() if sp.issparse(B) else (B * B).sum(1)

K_tr_blocks = [gram(b, b) for b in blocks_tr]
K_te_blocks = [gram(bt, b) for bt, b in zip(blocks_te, blocks_tr)]
d_tr = [diag_of(b) for b in blocks_tr]
d_te = [diag_of(b) for b in blocks_te]

def combine(w, Ks, d_rows, d_cols):
    K = sum(wk * Kk for wk, Kk in zip(w, Ks))
    dr = sum(wk * dk for wk, dk in zip(w, d_rows))
    dc = sum(wk * dk for wk, dk in zip(w, d_cols))
    den = np.sqrt(np.outer(dr, dc)); den[den == 0] = 1.0
    return K / den

W_DEFAULT = (1.0, 1.0, 1.0)
K_tr = combine(W_DEFAULT, K_tr_blocks, d_tr, d_tr)
K_te = combine(W_DEFAULT, K_te_blocks, d_te, d_tr)
print("K_tr", K_tr.shape, " K_te", K_te.shape)

K_tr (2404, 2404)  K_te (601, 2404)


## Validity (PSD) checks

`check_psd` reports, for a given kernel matrix:

- the symmetry error;
- the smallest eigenvalue, which must be nonnegative for a PSD matrix;
- whether a Cholesky factorization succeeds after adding a small diagonal term.

It is applied to each block, to the combined kernel, and to kernels with random nonnegative weights. Negative eigenvalues of magnitude about $10^{-12}$ are floating-point error. A thresholded similarity (1 if two points are within distance 1, otherwise 0) is included as a negative control. It has a negative eigenvalue and is expected to fail.

In [12]:
def check_psd(K, name, tol=1e-8):
    sym_err = np.abs(K - K.T).max()
    eig = np.linalg.eigvalsh((K + K.T) / 2)
    try:
        np.linalg.cholesky((K + K.T) / 2 + 1e-8 * np.eye(len(K))); chol = "ok"
    except np.linalg.LinAlgError:
        chol = "FAILED"
    verdict = "PSD" if eig.min() >= -tol else "NOT PSD"
    print(f"{name:34s} symmetry err {sym_err:.1e} | min eig {eig.min(): .2e} | Cholesky(+1e-8 I): {chol} -> {verdict}")
    return eig

for nm, Kb in zip(["stem", "affix", "char"], K_tr_blocks):
    check_psd(Kb, f"block K_{nm}")
eig_full = check_psd(K_tr, "full normalized kernel (1,1,1)")

G35 = np.array([[1, 1, 0], [1, 1, 1], [0, 1, 1]], float)
check_psd(G35, "thresholded similarity (negative control)")
c = np.array([1, -np.sqrt(2), 1]); print("  c^T G c =", c @ G35 @ c, "(= 4 - 4*sqrt(2) < 0)")

rng = np.random.default_rng(0)
for _ in range(3):
    w = rng.uniform(0, 2, 3)
    check_psd(combine(w, K_tr_blocks, d_tr, d_tr), f"random weights {np.round(w, 2)}")

block K_stem                       symmetry err 2.2e-16 | min eig  1.44e-02 | Cholesky(+1e-8 I): ok -> PSD
block K_affix                      symmetry err 4.4e-16 | min eig -3.74e-13 | Cholesky(+1e-8 I): ok -> PSD
block K_char                       symmetry err 0.0e+00 | min eig  7.78e-04 | Cholesky(+1e-8 I): ok -> PSD
full normalized kernel (1,1,1)     symmetry err 3.3e-16 | min eig  6.09e-03 | Cholesky(+1e-8 I): ok -> PSD
thresholded similarity (negative control) symmetry err 0.0e+00 | min eig -4.14e-01 | Cholesky(+1e-8 I): FAILED -> NOT PSD
  c^T G c = -1.6568542494923804 (= 4 - 4*sqrt(2) < 0)
random weights [1.27 0.54 0.08]    symmetry err 2.2e-16 | min eig  1.02e-02 | Cholesky(+1e-8 I): ok -> PSD
random weights [0.03 1.63 1.83]    symmetry err 3.3e-16 | min eig  7.02e-04 | Cholesky(+1e-8 I): ok -> PSD
random weights [1.21 1.46 1.09]    symmetry err 3.3e-16 | min eig  5.81e-03 | Cholesky(+1e-8 I): ok -> PSD


## Baselines

Three baselines are trained on the same preprocessed text:

- word TF-IDF with a linear SVM (bag-of-words kernel);
- character 3 to 5-gram TF-IDF with a linear SVM (contiguous n-grams, as in the spectrum kernel);
- hard bag-of-stems with a linear SVM, using the most probable stem of each word. This is the linear kernel on stem counts, so it shows what stem grouping alone achieves.

The regularization parameter is fixed at $C = 1$ for the baselines. The custom kernel selects $C$ by cross-validation (see the Cross-validation section).

In [13]:
def evaluate(name, y_true, y_pred, store):
    acc = accuracy_score(y_true, y_pred); f1 = f1_score(y_true, y_pred, average="macro")
    store[name] = (acc, f1); print(f"{name:42s} acc {acc:.4f}  macro-F1 {f1:.4f}")

results = {}

tv_w = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, min_df=MIN_DF, sublinear_tf=True)
Xw_tr = tv_w.fit_transform(docs_tr); Xw_te = tv_w.transform(docs_te)
evaluate("word TF-IDF + linear SVM", y_te, LinearSVC(C=1.0).fit(Xw_tr, y_tr).predict(Xw_te), results)

tv_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=MIN_DF, sublinear_tf=True)
Xch_tr = tv_c.fit_transform(docs_tr); Xch_te = tv_c.transform(docs_te)
evaluate("char 3-5-gram TF-IDF + linear SVM", y_te, LinearSVC(C=1.0).fit(Xch_tr, y_tr).predict(Xch_te), results)

best_row = np.asarray(M.argmax(0)).ravel()
M_hard = sp.csr_matrix((np.ones(V), (best_row, np.arange(V))), shape=(V, V))
Xs_tr = normalize(X_tr @ M_hard.T); Xs_te = normalize(X_te @ M_hard.T)
evaluate("hard bag-of-stems + linear SVM", y_te, LinearSVC(C=1.0).fit(Xs_tr, y_tr).predict(Xs_te), results)

word TF-IDF + linear SVM                   acc 0.9617  macro-F1 0.9617
char 3-5-gram TF-IDF + linear SVM          acc 0.9501  macro-F1 0.9501
hard bag-of-stems + linear SVM             acc 0.9484  macro-F1 0.9484


## Cross-validation

Five-fold stratified cross-validation selects the block weights (`WEIGHTS`) and the regularization parameter $C$ (`C_GRID`). Each fold uses `SVC(kernel="precomputed")`: training uses `G[train, train]` and validation uses `G[validation, train]`. The combination with the highest mean accuracy is selected.

In [14]:
WEIGHTS = [(1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 0), (1, 0, 1), (1, 1, 1), (2, 1, 1)]
C_GRID = [0.1, 1, 10, 100]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(np.zeros(len(y_tr)), y_tr))

def cv_score(K, C):
    accs = []
    for tr, va in folds:
        clf = SVC(kernel="precomputed", C=C).fit(K[np.ix_(tr, tr)], y_tr[tr])
        accs.append(accuracy_score(y_tr[va], clf.predict(K[np.ix_(va, tr)])))
    return float(np.mean(accs))

cv_table = {}
for w in WEIGHTS:
    Kw = combine(w, K_tr_blocks, d_tr, d_tr)
    for C in C_GRID:
        cv_table[(w, C)] = cv_score(Kw, C)
    best_c = max(C_GRID, key=lambda C: cv_table[(w, C)])
    print(f"weights {str(w):10s} best C = {best_c:<6} CV acc = {cv_table[(w, best_c)]:.4f}")

best_w, best_C = max(cv_table, key=cv_table.get)
print("\nbest:", best_w, "C =", best_C, "CV acc =", round(cv_table[(best_w, best_C)], 4))

weights (1, 0, 0)  best C = 10     CV acc = 0.9592
weights (0, 1, 0)  best C = 100    CV acc = 0.7292
weights (0, 0, 1)  best C = 1      CV acc = 0.9305
weights (1, 1, 0)  best C = 10     CV acc = 0.9551
weights (1, 0, 1)  best C = 10     CV acc = 0.9542
weights (1, 1, 1)  best C = 10     CV acc = 0.9538
weights (2, 1, 1)  best C = 10     CV acc = 0.9567

best: (1, 0, 0) C = 10 CV acc = 0.9592


## Final model, ablations and RBF wrapper

- The model is refit on the full training set with the selected weights and $C$, and evaluated once on the test set. A classification report is printed.
- Ablations train the stem, affix and character blocks separately, each with its own cross-validated $C$.
- The RBF wrapper uses $K_{\text{rbf}} = \exp\!\bigl(-\gamma\,(2 - 2\tilde{K})\bigr)$ for the normalized kernel $\tilde{K}$, which is valid because $\lVert \Phi(x) - \Phi(y) \rVert^{2} = 2 - 2\tilde{K}(x, y)$ when the diagonal equals 1. $\gamma$ and $C$ are selected by cross-validation.
- The explicit-feature check trains a linear SVM on the concatenated block features and prints its agreement with the precomputed-kernel SVM. They should agree, because the kernel is an explicit inner product.

In [15]:
def fit_predict(w, C, kernel_wrapper=None):
    Ktr = combine(w, K_tr_blocks, d_tr, d_tr); Kte = combine(w, K_te_blocks, d_te, d_tr)
    if kernel_wrapper: Ktr, Kte = kernel_wrapper(Ktr), kernel_wrapper(Kte)
    return SVC(kernel="precomputed", C=C).fit(Ktr, y_tr).predict(Kte)

pred = fit_predict(best_w, best_C)
evaluate(f"custom kernel {best_w}, C={best_C}", y_te, pred, results)
print(classification_report(y_te, pred, digits=4))

for nm, w in [("stem only", (1, 0, 0)), ("affix only", (0, 1, 0)), ("char-subsequence only", (0, 0, 1))]:
    C_b = max(C_GRID, key=lambda C: cv_table[(w, C)])
    evaluate(f"ablation: {nm}, C={C_b}", y_te, fit_predict(w, C_b), results)

K_best = combine(best_w, K_tr_blocks, d_tr, d_tr)
rbf_cv = {}
for g in [0.25, 0.5, 1.0, 2.0]:
    Kg = np.exp(-g * (2 - 2 * K_best))
    for C in C_GRID:
        rbf_cv[(g, C)] = cv_score(Kg, C)
g_best, C_rbf = max(rbf_cv, key=rbf_cv.get)
print(f"\nRBF wrapper: best gamma = {g_best}, C = {C_rbf}, CV acc = {rbf_cv[(g_best, C_rbf)]:.4f}")
pred_rbf = fit_predict(best_w, C_rbf, kernel_wrapper=lambda K: np.exp(-g_best * (2 - 2 * K)))
evaluate(f"custom kernel + RBF wrapper (gamma={g_best})", y_te, pred_rbf, results)

Ztr = normalize(sp.hstack([np.sqrt(wk) * b for wk, b in zip(best_w, blocks_tr) if wk > 0]).tocsr())
Zte = normalize(sp.hstack([np.sqrt(wk) * b for wk, b in zip(best_w, blocks_te) if wk > 0]).tocsr())
lin = SVC(kernel="linear", C=best_C).fit(Ztr, y_tr)
print("\nTest-set agreement between the explicit-feature linear SVC and the precomputed-kernel SVC:",
      np.mean(lin.predict(Zte) == pred))

custom kernel (1, 0, 0), C=10              acc 0.9517  macro-F1 0.9517
              precision    recall  f1-score   support

           0     0.9412    0.9632    0.9521       299
           1     0.9627    0.9404    0.9514       302

    accuracy                         0.9517       601
   macro avg     0.9519    0.9518    0.9517       601
weighted avg     0.9520    0.9517    0.9517       601

ablation: stem only, C=10                  acc 0.9517  macro-F1 0.9517
ablation: affix only, C=100                acc 0.7005  macro-F1 0.6999
ablation: char-subsequence only, C=1       acc 0.9235  macro-F1 0.9235

RBF wrapper: best gamma = 0.25, C = 10, CV acc = 0.9596
custom kernel + RBF wrapper (gamma=0.25)   acc 0.9517  macro-F1 0.9517

Test-set agreement between the explicit-feature linear SVC and the precomputed-kernel SVC: 1.0


## Results

Prints the test accuracy and macro-F1 of every model trained above.

In [16]:
print(f"{'model':46s} {'acc':>7s} {'macro-F1':>9s}")
for k, (a, f) in results.items():
    print(f"{k:46s} {a:7.4f} {f:9.4f}")

model                                              acc  macro-F1
word TF-IDF + linear SVM                        0.9617    0.9617
char 3-5-gram TF-IDF + linear SVM               0.9501    0.9501
hard bag-of-stems + linear SVM                  0.9484    0.9484
custom kernel (1, 0, 0), C=10                   0.9517    0.9517
ablation: stem only, C=10                       0.9517    0.9517
ablation: affix only, C=100                     0.7005    0.6999
ablation: char-subsequence only, C=1            0.9235    0.9235
custom kernel + RBF wrapper (gamma=0.25)        0.9517    0.9517


**Limitations**

1. Words outside the training vocabulary are ignored at test time.
2. The stemmer is rule-based. Reduplication of multi-syllable roots, vowel changes other than u/o, and Taglish compounds are only partly covered. A real word that is a substring of another word can also cause over-segmentation (for example, *pagod* reduced to *god*).
3. The IDF weights and root lexicon are estimated on all training documents, including those in validation folds. The estimation is unsupervised, so the effect is expected to be small.
4. Labels are assigned at the level of the source publication, so accuracy may partly reflect publisher style.
5. A single train/test split cannot resolve accuracy differences of a few documents. Repeated cross-validation and a paired test are recommended.